In [9]:
from pyspark.sql import SparkSession

In [10]:
# Instantiate spark session
spark = SparkSession.builder \
    .appName("Walmart SQL Analysis") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()


In [11]:
# Load CSV as DataFrame
df = spark.read.csv("walmart_transformed_real.csv", header=True, inferSchema=True)

# Show the first few rows
df.show()


+-----+----+-----+---+------------+------------+-----------+----------+-----------+------------+-------------+--------------------+-----------+-----------------+
|Month|Year|Store|Day|Weekly_Sales|Holiday_Flag|Temperature|Fuel_Price|        CPI|Unemployment|FormattedDate|FormattedHolidayFlag|HolidayType|Real_Weekly_Sales|
+-----+----+-----+---+------------+------------+-----------+----------+-----------+------------+-------------+--------------------+-----------+-----------------+
|    2|2010|    8| 26|   847592.11|           0|      37.91|     2.561|214.6940735|       6.299|    26-Feb-10|         Not Holiday|Non-Holiday|      394790.6415|
|    9|2010|   16|  3|   542087.89|           0|      58.02|     2.773|190.3621607|       6.868|     3-Sep-10|         Not Holiday|Non-Holiday|      284766.6196|
|    9|2010|   36|  3|   431294.45|           0|      82.29|     2.533|210.2966631|        8.36|     3-Sep-10|         Not Holiday|Non-Holiday|      205088.5847|
|    9|2010|   17|  3|   834

In [12]:
# Register csv as a table
df.createOrReplaceTempView("sales_data")


In [13]:
top_5_store_sales = spark.sql("""
    SELECT Store, SUM(Weekly_Sales) as total_sales
    FROM sales_data
    GROUP BY store
    ORDER BY total_sales DESC
    LIMIT 5
""")

top_5_store_sales.show()

+-----+--------------------+
|Store|         total_sales|
+-----+--------------------+
|   20|      3.0139779246E8|
|    4| 2.995439533799999E8|
|   14|2.8899991134000003E8|
|   13|       2.865177038E8|
|    2|      2.7538244098E8|
+-----+--------------------+



Here we can see the top 5 most productive stores over our time frame. We can see store 20 produced about $300 million in total sales from 2010-2012.

In [14]:
bottom_5_store_sales = spark.sql("""
    SELECT Store, SUM(Weekly_Sales) as total_sales
    FROM sales_data
    GROUP BY store
    ORDER BY total_sales ASC
    LIMIT 5
""")

bottom_5_store_sales.show()

+-----+--------------------+
|Store|         total_sales|
+-----+--------------------+
|   33|3.7160221960000016E7|
|   44| 4.329308784000002E7|
|    5|4.5475688900000006E7|
|   36| 5.341221496999998E7|
|   38| 5.515962642000002E7|
+-----+--------------------+



Here are the total sales for the bottom 5 stores. Store 33 only produced about $37 million in sales.

In [15]:
total_sales = spark.sql("""
    SELECT SUM(Weekly_Sales) AS total_sales
    FROM sales_data
""")

total_sales.show()

+-------------------+
|        total_sales|
+-------------------+
|6.737218987109989E9|
+-------------------+



Here are the total sales over the time period. From 2010-2012, all stores produced a total of about $6.73 billion in sales.

In [18]:
store_20 = spark.sql("""
    SELECT Year, SUM(Real_Weekly_Sales)
    FROM sales_data
    WHERE Store = 20 AND Month >= 2 AND Month <= 9
    GROUP BY Year
""")

store_20.show()

+----+----------------------+
|Year|sum(Real_Weekly_Sales)|
+----+----------------------+
|2012|       3.44413462793E7|
|2010|   3.384310652150001E7|
|2011|  3.4302199809999995E7|
+----+----------------------+



We want to be able to study how different factors such as CPI, temperature, and unemployment factor into sales. We've already tried to make aggregated visualizations on how these variables affect sales, but we want to try and remove as many confounding variables as possible. We filter out all records except those of store 20, our top store. While the information we learn from store 20 might not always be generalizable, we need to be able to see how individual stores are affected, and it makes sense to make decisions off of our most lucrative stores.

By taking the common period from February to September, we can try to compare the total sales for store 20 year over year. It appears that the amount of sales has remained relatively constant over time with no real growth.

In [19]:
spark.stop()